# Sovereign Bond Pricing Model Analysis

This notebook evaluates BONTAM fair values under several short-rate models, using historical TAMAR data and an internal TAMAR forward view.

### Objectives
- Build the observed and forward TAMAR path
- Simulate future short-rate trajectories
- Estimate terminal values for BONTAM bonds
- Compare trading prices with model-implied fair values
- Infer the market price of uncertainty from observed pricing gaps

In [ ]:
from datetime import date

# =========================
# Global Configuration
# =========================

EMISSION_DATE = date(2025, 1, 29)
TAMAR_START_DATE = date(2025, 1, 15)   # average starts counting 10 business days before emission

# Simulation parameters
M = 10_000
SIGMA = 0.1428
ALPHA = 0.5
THETA = 0.21
KAPPA = 0.5
SEED = 123

# Model choice: 'CIR', 'bk', 'hw', or 'basic'
INTEREST_SIMULATION_TYPE = "CIR"

In [ ]:
# =========================
# Imports
# =========================

import io
import math as mt
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import numpy as np
import pandas as pd
import requests
import seaborn as sns

from datetime import date
from pandas.tseries.offsets import CustomBusinessDay
from scipy.optimize import minimize, minimize_scalar
from scipy.sparse import diags
from scipy.sparse.linalg import spsolve
from scipy.stats import ncx2

warnings.filterwarnings("ignore", category=RuntimeWarning)
np.random.seed(SEED)

In [ ]:
# =========================
# Plot Styling
# =========================

plt.rcParams["font.family"] = "Tahoma"
plt.rcParams["figure.figsize"] = (12, 6)
plt.rcParams["axes.grid"] = True
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

NAVY = (0 / 255, 49 / 255, 77 / 255)
BLUE = (0 / 255, 121 / 255, 204 / 255)
GREY = (165 / 255, 165 / 255, 165 / 255)

In [ ]:
# =========================
# Argentina Business Day Calendar
# =========================

ARG_HOLIDAYS_DT = pd.to_datetime(ARGENTINA_HOLIDAYS)
ARG_BD = CustomBusinessDay(holidays=ARG_HOLIDAYS_DT)

## 1. Historical TAMAR and Forward View

This section loads the historical TAMAR series, appends the in-house TAMAR forward view, and visualizes the combined path used for simulations.

In [ ]:
from TAMAR_api import get_tamar_daily
from TAMAR_view import TAMAR_view

# Load historical TAMAR
tamar = get_tamar_daily()
observed_TAMAR_series = tamar["TAMAR"][TAMAR_START_DATE:] / 100
observed_TAMAR = observed_TAMAR_series.values
most_recent_tamar_date = observed_TAMAR_series.index[-1]
observed_TAMAR_df = pd.DataFrame(observed_TAMAR_series)

# Align in-house TAMAR view with realized data
tamars_realized_since_setting_tamar_view = get_distance_days_252(
    "2025-10-20",
    most_recent_tamar_date
)

TAMAR_view = TAMAR_view[tamars_realized_since_setting_tamar_view:]

# Create indexed forward TAMAR view
tamar_view_start_date = observed_TAMAR_df.index[-1] + ARG_BD
tamar_view_dates = pd.bdate_range(
    start=tamar_view_start_date,
    periods=len(TAMAR_view),
    freq=ARG_BD
)

TAMAR_view_series_indexed = pd.Series(TAMAR_view, index=tamar_view_dates)

# Combine historical and forward series
combined_historical_and_view = pd.concat([
    observed_TAMAR_df,
    pd.DataFrame(TAMAR_view_series_indexed, columns=["TAMAR"])
])

In [ ]:
fig, ax = plt.subplots()

ax.plot(
    np.arange(len(observed_TAMAR_df)),
    observed_TAMAR_df.values,
    label="Historical TAMAR",
    color=NAVY
)

ax.plot(
    np.arange(len(observed_TAMAR_df), len(combined_historical_and_view)),
    TAMAR_view_series_indexed.values,
    label="TAMAR View",
    color=NAVY,
    linestyle="--"
)

ax.set_title("Historical TAMAR and TAMAR View", loc="left", fontweight="bold")
ax.set_xlabel("Business Day Number")
ax.set_ylabel("TAMAR Rate")
ax.legend()

plt.show()

## 2. Short-Rate Simulation

The following section simulates future TAMAR paths under the selected short-rate model and appends them to the observed historical series.

In [ ]:
from bk_model import simulate_BK_trajectories
from cir_model import simulate_cir_trajectories
from hull_white import simulate_HW_trajectories
from simple_diffusion import simulate_trajectories

r0 = float(observed_TAMAR[-1])
N = 287 - tamars_realized_since_setting_tamar_view
T_years = N / 252

basic = simulate_trajectories(
    sigma=SIGMA, r0=r0, T_years=T_years, N=N, M=M
)
CIR = simulate_cir_trajectories(
    sigma=SIGMA, kappa=KAPPA, theta=THETA, r0=r0, T_years=T_years, N=N, M=M
)
hw = simulate_HW_trajectories(
    r0=r0, alpha=ALPHA, sigma=SIGMA, TAMAR_view=TAMAR_view, M=M, T_years=T_years, N=N
)
bk = simulate_BK_trajectories(
    r0=r0, alpha=ALPHA, sigma=SIGMA, TAMAR_view=TAMAR_view, M=M, T_years=T_years, N=N
)

if INTEREST_SIMULATION_TYPE == "CIR":
    simulation_type = CIR
elif INTEREST_SIMULATION_TYPE == "hw":
    simulation_type = hw
elif INTEREST_SIMULATION_TYPE == "bk":
    simulation_type = bk
elif INTEREST_SIMULATION_TYPE == "basic":
    simulation_type = basic
else:
    raise ValueError("Please choose a valid simulation type.")

In [ ]:
last_observed_date = observed_TAMAR_df.index[-1]
sim_start_date_for_index = last_observed_date + ARG_BD
sim_dates = pd.bdate_range(start=sim_start_date_for_index, periods=N, freq=ARG_BD)

sim_df_indexed = pd.DataFrame(
    simulation_type[1:, :],
    index=sim_dates,
    columns=[f"Path_{i}" for i in range(M)]
)

historical_expanded = pd.DataFrame({
    f"Path_{i}": observed_TAMAR_df.iloc[:, 0]
    for i in range(M)
})

combined_df = pd.concat([historical_expanded, sim_df_indexed])

fig, ax = plt.subplots()

ax.plot(np.arange(len(combined_df)), combined_df.values, color=NAVY, linewidth=1)
ax.set_title(
    f"Figure 1. {INTEREST_SIMULATION_TYPE} Simulated and Historical TAMAR Paths",
    loc="left",
    fontweight="bold"
)
ax.set_xlabel("Business Days Since Issuance Accrual Average")
ax.set_ylabel("TAMAR Rate")
ax.yaxis.set_major_formatter(mtick.PercentFormatter(xmax=1, decimals=0))

plt.show()

## 3. Bond Valuation

This section computes risk-neutral terminal values and discounted fair values for each BONTAM bond under the selected rate simulation.

In [ ]:
def evaluate_bond(bond_expiry, base_rate, discount_rate, trading_price, bond_name, figure_number):
    expected_tv = trading_price / discounted_value(1, discount_rate, bond_expiry)
    terminal_values = get_terminal_value(bond_expiry, base_rate, simulation_type)

    discounted_vpv, variance, non_discount_option_value, vpv_with_floor = discounted_data(
        terminal_values,
        discount_rate,
        bond_expiry,
        trading_price,
        bond_name,
        figure_number
    )

    return {
        "bond_name": bond_name,
        "bond_expiry": bond_expiry,
        "base_rate": base_rate,
        "discount_rate": discount_rate,
        "trading_price": trading_price,
        "expected_tv": expected_tv,
        "discounted_vpv": discounted_vpv,
        "variance": variance,
        "non_discount_option_value": non_discount_option_value,
        "vpv_with_floor": vpv_with_floor
    }

In [ ]:
bond_results = [
    evaluate_bond(
        bond_expiry=pd.to_datetime("2026-03-16"),
        base_rate=0.0225,
        discount_rate=discount_rate_table[1],
        trading_price=TTM26_price,
        bond_name="TTM26",
        figure_number="4"
    ),
    evaluate_bond(
        bond_expiry=pd.to_datetime("2026-06-30"),
        base_rate=0.0219,
        discount_rate=discount_rate_table[2],
        trading_price=TTJ26_price,
        bond_name="TTJ26",
        figure_number="5"
    ),
    evaluate_bond(
        bond_expiry=pd.to_datetime("2026-09-15"),
        base_rate=0.0217,
        discount_rate=discount_rate_table[3],
        trading_price=TTS26_price,
        bond_name="TTS26",
        figure_number="6"
    ),
    evaluate_bond(
        bond_expiry=pd.to_datetime("2026-12-15"),
        base_rate=0.0214,
        discount_rate=discount_rate_table[4],
        trading_price=TTD26_price,
        bond_name="TTD26",
        figure_number="7"
    )
]

## 4. Market Price of Uncertainty

We estimate the market price of uncertainty by comparing non-discounted terminal values, expected terminal values, and the variance of bond payments.

In [ ]:
bdays_till_expiry = [60, 131, 186, 246]
trading_prices = [x["trading_price"] for x in bond_results]
risk_neutral_non_discount_option = [x["non_discount_option_value"] for x in bond_results]
risk_neutral_fair_values = [x["discounted_vpv"] for x in bond_results]
expected_tvs = [x["expected_tv"] for x in bond_results]
variances_in_payment = [x["variance"] for x in bond_results]
non_discounted_terminal_values = [x["vpv_with_floor"] for x in bond_results]

difference_in_view = [
    fair_value - trading_price
    for fair_value, trading_price in zip(risk_neutral_fair_values, trading_prices)
]

estimated_risk_aversion_coefficient = [
    (non_discounted_terminal_values[i] - expected_tvs[i]) / variances_in_payment[i]
    for i in range(4)
]

optimal_risk_aversion = np.mean(estimated_risk_aversion_coefficient)

print("Estimated risk aversion coefficients:", estimated_risk_aversion_coefficient)
print("Optimal risk aversion:", optimal_risk_aversion)

In [ ]:
expiry_lookup_table = {
    1: "2026-03-16",
    2: "2026-06-30",
    3: "2026-09-15",
    4: "2026-12-15"
}

risk_averse_no_discount_prices = [
    non_discounted_terminal_values[i] - optimal_risk_aversion * variances_in_payment[i]
    for i in range(4)
]

risk_averse_discounted_prices = [
    discounted_value(risk_averse_no_discount_prices[i], discount_rate_table[i + 1], expiry_lookup_table[i + 1])
    for i in range(4)
]

In [ ]:
results_df = pd.DataFrame({
    "Bond Name": [
        "BONTAM Mar-26 (TTM26)",
        "BONTAM Jun-26 (TTJ26)",
        "BONTAM Sep-26 (TTS26)",
        "BONTAM Dec-26 (TTD26)"
    ],
    "TAMAR Observations Left to Accrue": bdays_till_expiry,
    "Trading Price": trading_prices,
    "Puente Risk-Neutral Fair Value": risk_neutral_fair_values,
    "Puente Risk-Averse Discounted Price": risk_averse_discounted_prices
}).set_index("Bond Name")

styled_results = (
    results_df.style
    .set_table_styles([
        {
            "selector": "th",
            "props": [
                ("background-color", "#07203C"),
                ("color", "white"),
                ("font-size", "12pt"),
                ("font-weight", "bold"),
                ("font-family", "Tahoma"),
                ("text-align", "center")
            ]
        },
        {
            "selector": "td",
            "props": [
                ("background-color", "white"),
                ("font-family", "Tahoma"),
                ("color", "black")
            ]
        },
        {
            "selector": "th.row_heading",
            "props": [
                ("background-color", "#D9D9D9"),
                ("font-family", "Tahoma"),
                ("color", "black")
            ]
        }
    ])
    .format(precision=3)
)

display(styled_results)

In [ ]:
fig, ax = plt.subplots()

ax.plot(bdays_till_expiry, trading_prices, color=BLUE, linestyle="--", label="Trading Price")
ax.plot(bdays_till_expiry, risk_neutral_fair_values, color=NAVY, label="Risk-Neutral Fair Value")

ax.set_title(
    "Figure 2. Puente BONTAM Fair Values Based on In-House Model",
    loc="left",
    fontweight="bold"
)
ax.set_xlabel("TAMAR Observations Left to Accrue")
ax.set_ylabel("Price in ARS")
ax.legend()

plt.show()

In [ ]:
fig, ax = plt.subplots()

ax.plot(bdays_till_expiry, risk_averse_discounted_prices, color=GREY, label="Risk-Averse Discounted Price")
ax.plot(bdays_till_expiry, trading_prices, color=BLUE, linestyle="--", label="Trading Price")
ax.plot(bdays_till_expiry, risk_neutral_fair_values, color=NAVY, label="Risk-Neutral Fair Value")

ax.set_title(
    "Figure 3. Puente BONTAM Fair Values with Risk Adjustment",
    loc="left",
    fontweight="bold"
)
ax.set_xlabel("TAMAR Observations Left to Accrue")
ax.set_ylabel("Price in ARS")
ax.legend()

plt.show()

## 5. Summary

The model-implied risk-neutral fair values exceed observed trading prices across the BONTAM curve. Interpreting this gap through a mean-variance adjustment implies a positive market price of uncertainty, suggesting that investors require compensation for exposure to TAMAR-linked payment volatility.